In [12]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# 1. Load the sequence dataset [cite: 26]
dataset = """artificial intelligence systems learn patterns from data.
sequence models process information step by step.
recurrent neural networks are useful for sequence prediction.
1stm networks handle long term dependencies.
deep learning models improve sequence learning.
generative models create new samples from learned patterns.
language models predict the next word in a sentence.
sequence generation is used in chatbots and assistants.
machine learning helps computers learn automatically.
training data improves model accuracy.
neural networks simulate human brain structures.
optimization algorithms improve learning efficiency.
technology is transforming modern education.
online learning platforms use artificial intelligence.
students benefit from intelligent tutoring systems.
automation improves productivity and decision making."""

In [8]:
# 2. Perform word-level tokenization 
tokenizer = Tokenizer()
corpus = dataset.lower().split('\n')
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1

# 3. Create input-output sequence pairs 
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences so they are all the same length
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Split into predictors (X) and label (y)
X, y = input_sequences[:,:-1], input_sequences[:,-1]
# Convert the label to one-hot encoding
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print(f"Total Words in Vocabulary: {total_words}")
print(f"Max Sequence Length: {max_sequence_len}")
print(f"Total Input Sequences: {len(input_sequences)}")

Total Words in Vocabulary: 78
Max Sequence Length: 9
Total Input Sequences: 88


In [9]:
from tensorflow.keras.layers import LSTM

# 4. Design an LSTM based generative model
lstm_model = Sequential()
lstm_model.add(Embedding(total_words, 10, input_length=max_sequence_len-1))
# Here is the upgrade: Swapping SimpleRNN for LSTM
lstm_model.add(LSTM(50))
lstm_model.add(Dense(total_words, activation='softmax'))

lstm_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_model.summary()

# 5. Train the LSTM model
print("\nStarting LSTM training...")
lstm_history = lstm_model.fit(X, y, epochs=100, verbose=1)
print("LSTM Training complete!")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Starting LSTM training...
Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.0114 - loss: 4.3572  
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0568 - loss: 4.3537
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0682 - loss: 4.3509
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0568 - loss: 4.3475
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0568 - loss: 4.3443
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0568 - loss: 4.3402
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0568 - loss: 4.3352    
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0568 - loss: 4.3294
Epoch 9/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0568 - loss: 4.3213
Epoch 10/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0568 - loss: 4.3096
Epoch 11/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0568 - loss: 4.2969
Epoch 12/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [14]:
import numpy as np

# 6. Generate new sequences using a seed input (Updated with Temperature)
def generate_sequence(seed_text, next_words, model, max_sequence_len, temperature=1.0):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        
        # Predict the probabilities for the next word
        # We add [0] at the end to grab the 1D array of probabilities for the current step
        predicted_probs = model.predict(token_list, verbose=0)[0] 
        
        # --- The Sampling Upgrade ---
        # 1. Adjust the probabilities using the "temperature"
        # We add a tiny number (1e-7) to avoid taking the log of 0
        preds = np.log(predicted_probs + 1e-7) / temperature
        exp_preds = np.exp(preds)
        predicted_probs = exp_preds / np.sum(exp_preds) # Normalize back to 100%
        
        # 2. Roll the dice! Pick an index based on the weighted probabilities
        predicted_index = np.random.choice(len(predicted_probs), p=predicted_probs)
        # ----------------------------
        
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break
        
        # Only append if a valid word was found
        if output_word:
            seed_text += " " + output_word
            
    return seed_text

In [15]:
# Let's test it with a few seed phrases!
seed_1 = "machine learning"
seed_2 = "generative models"

# 7. Generate sequences with the new model
print("\n--- LSTM Generated Sequences ---")
print("\n--- Low Temperature (0.2) - Safe & Repetitive ---")
print(generate_sequence(seed_1, 20, lstm_model, max_sequence_len, temperature=0.2))

print("\n--- Medium Temperature (0.7) - Balanced ---")
print(generate_sequence(seed_1, 20, lstm_model, max_sequence_len, temperature=0.7))

print("\n--- High Temperature (1.5) - Chaotic & Creative ---")
print(generate_sequence(seed_1, 20, lstm_model, max_sequence_len, temperature=1.5))


--- LSTM Generated Sequences ---

--- Low Temperature (0.2) - Safe & Repetitive ---
machine learning learning learning learning intelligence intelligence patterns patterns learning intelligence patterns from patterns patterns from learning learning patterns intelligence patterns intelligence

--- Medium Temperature (0.7) - Balanced ---
machine learning improve models algorithms learning from improve intelligence intelligence improve learning systems learning efficiency patterns from new learning automatically efficiency learning

--- High Temperature (1.5) - Chaotic & Creative ---
machine learning new models from learning sequence sequence helps education from decision term by sentence prediction new improve structures sentence accuracy machine
